This script was used to add the calculation of the bounds etc. to the anndata object, as it is used so often that it seems smart to safe the values to safe computational time. 

In [1]:
import numpy as np
import yaml
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import anndata as ad
import seaborn as sns
import pandas as pd
import scanpy as sc

from tqdm import tqdm
from velotest.hypothesis_testing import run_hypothesis_test_on
from velotest.neighbors import find_neighbors_in_direction_of_velocity, find_neighbors

In [2]:
def calculateBound(adata, k=None, neighbors=True, number_neighbors=200): 
    if neighbors is True:
        nn_indices_optimal = find_neighbors(torch.tensor(adata.layers['Ms']), k_neighbors=number_neighbors)

    mins = torch.tensor([-2.]*len(adata))
    maxs = torch.tensor([-2.]*len(adata))
    for i in tqdm(range(len(adata))):
        cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
        if neighbors is True:
            expression_all = torch.tensor(adata.layers['Ms'][nn_indices_optimal[i]][:,adata.var.velocity_genes])
        else:
            expression_all = torch.tensor(adata.layers['Ms'][:,adata.var.velocity_genes])
        expression_cell = torch.tensor(adata.layers['Ms'][i,adata.var.velocity_genes]).unsqueeze(0)
        velocity = torch.tensor(adata.layers['velocity'][i,adata.var.velocity_genes]).unsqueeze(0)
        cosines = cos(expression_all - expression_cell, velocity)
        cosines = torch.cat([cosines[:i], cosines[i+1:]])
        if k is None:
            mins[i] = torch.min(cosines)
            maxs[i] = torch.max(cosines)
        else:
            mins[i] = torch.mean(torch.topk(cosines, largest=False, k=k).values)
            maxs[i] = torch.mean(torch.topk(cosines, largest=True, k=k).values)

    return maxs

def bestVelocityEmbedded(adata, basis, test_neighbors = 200):
    uncorrected_p_values, h0_rejected, debug_dict = run_hypothesis_test_on(adata, basis=basis, number_neighbors_to_sample_from=test_neighbors, null_distribution="velocities-explicit", cosine_empty_neighborhood=None, correction="bonferroni", restrict_to_velocity_genes=True, exclusion_degree=10)
    statistics = debug_dict['statistics']
    # Added this line, because some cells still had a statistice but in regard to the p_value they were empty neighborhoods.
    for i, p_val in enumerate(uncorrected_p_values):
            if p_val == 2:
                statistics[i] = None
    non_empty_neighborhoods_indices = np.where(uncorrected_p_values != 2)[0]

    # Calculate emmbeding test optimal statistic
    best_value = [statistics.get_max_value(offset=True)[1] for statistics in statistics if statistics is not None]
    original_value = [statistics.get_values()[0] for statistics in statistics if statistics is not None]

    return best_value, original_value, non_empty_neighborhoods_indices

In [9]:
datasets = {
        #"Pancreas_dynamical": "../configs/pancreas_dynamic.yaml",
        #"Pancreas_stochastic": "../configs/pancreas_stochastic.yaml",
        #"Dentate_gyrus": "../configs/dentategyrus.yaml",
        #"Gastrulation_erythroid": "../configs/gastrulation.yaml",
        #"Bone_marrow": "../configs/bonemarrow.yaml",
        #"PBMC": "../configs/pbmc.yaml",
        #"Covid": "../configs/covid.yaml",
        #"Retinal_development": "../configs/retinal_development.yaml",
        #"Human_HSC": "../configs/humanHSC.yaml",
        "RPE_1": "../configs/noEmbedding/intestinal_organoids.yaml",
        #"Hindbrain_mouse": "../configs/noEmbedding/hindbrain.yaml",
        #"Oligodendrocyte_mouse": "../configs/noEmbedding/oligodendrocyte.yaml",
        #"Bone_marrow_mouse": "../configs/noEmbedding/mouse_bonemarrow.yaml",
        #"Cerebral_cortex_human": "../configs/noEmbedding/human_cerebral.yaml",
        #"Hematopoiesis_mouse": "../configs/noEmbedding/mouse_hema.yaml",
        #"Spermatogenisis": "../configs/noEmbedding/spermatogenesis.yaml",
        #"Old_brain": "../configs/noEmbedding/old_brain.yaml",
        #"Prefrontal_cortex": "../configs/noEmbedding/pfc.yaml",
        #"Intestinal_organoids":"../configs/noEmbedding/GSE128365.yaml",
        #"Zebra_NMP": "../configs/noEmbedding/zebra_NMP.yaml"
    }

In [10]:
for name, path in datasets.items():
    with open(path, "r") as f:
        config = yaml.safe_load(f)

    # Collect data information
    adata = sc.read(config['path'])
    basis = config['basis']
    neighbors = config['number_neighbors_to_sample_from']
    cluster = config['cluster']
    color = config['cluster_colors']

    general = calculateBound(adata, neighbors=False)
    upper_neigh = calculateBound(adata, neighbors=True, number_neighbors=neighbors)
    best, original, non_empty_indices = bestVelocityEmbedded(adata, basis=basis, test_neighbors=neighbors)

    adata.obsm['general_bound'] = general.numpy()
    adata.obsm['upper_max_neighbors'] = upper_neigh.numpy()
    adata.uns['non_empty_indices'] = non_empty_indices
    adata.uns['best_embedding'] = best
    adata.uns['original_cosine_value'] = original
    adata.write_h5ad(config['path'])

100%|██████████| 5422/5422 [04:03<00:00, 22.26it/s]


Dropping all the genes which are not velocity genes. 387 genes left.
Number of workers:  9
Runnining with new modification


100%|██████████| 5422/5422 [00:34<00:00, 155.32it/s]


Exclusion radius in rad: 0.17453292519943295
